# 1. Setup and Installation


In [ ]:
!apt-get update && apt-get install -y espeak-ng
!pip install datasets torchaudio silero-vad librosa pyloudnorm indic_numtowords soundfile tqdm pandas pyarrow


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,721 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,306 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-se

# 2. Mount Google Drive
This is necessary to access the Rasa dataset folder and save the output files.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import os

def find_parquet_file(search_name="kathbath", search_path="/content/drive"):
    if not os.path.exists(search_path):
        print("Google Drive is not mounted! Please run Cell 2 first.")
        return

    found = False
    for root, dirs, files in os.walk(search_path):
        for file in files:
            if search_name.lower() in file.lower() and file.endswith(".parquet"):
                print(f"FOUND: {os.path.join(root, file)}")
                found = True
    if not found:
        print(f"Could not find any Parquet files containing '{search_name}' in Google Drive.")

find_parquet_file()


FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part1.parquet
FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part2.parquet
FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part3.parquet
FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part4.parquet
FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part5.parquet
FOUND: /content/drive/MyDrive/Kathbath_dataset/kathbath_hindi_30k_part6.parquet


# 3. Text Normalizer
This is the custom `HindiNormalizer` to handle text cleaning.


In [ ]:
import re
import html
import unicodedata
import logging
from indic_numtowords import num2words

logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class HindiNormalizer:
    def __init__(self):
        self.deva_digit_map = str.maketrans('०१२३४५६७८९', '0123456789')

        self.abbrev_dict = {
            'ISRO': 'इसरो', 'NASA': 'नासा', 'UNESCO': 'यूनेस्को', 'WHO': 'डब्लूएचओ',
            'UN': 'यूएन', 'USA': 'यूएसए', 'UK': 'यूके'
        }

        self.hindi_dot_abbrevs = {
            'डॉ': 'डॉक्टर', 'प्रो': 'प्रोफ़ेसर', 'सं': 'संख्या', 'पृ': 'पृष्ठ',
            'रु': 'रुपये', 'कि.मी': 'किलोमीटर', 'मी': 'मीटर', 'कि.ग्रा': 'किलोग्राम',
            'ई': 'ईसवी', 'ई.पू': 'ईसा पूर्व', 'वि.सं': 'विक्रम संवत'
        }

        self.url_context_words = {'वेबसाइट', 'लिंक', 'पता', 'यहाँ', 'जाएं', 'जाकर', 'देखें', 'क्लिक', 'संपर्क'}
        self.otp_context_words = {'OTP', 'ओटीपी', 'कोड', 'पिन', 'वेरिफिकेशन', 'सत्यापन'}

        self.domain_dict = {'gmail': 'जीमेल', 'yahoo': 'याहू', 'hotmail': 'हॉटमेल'}
        self.tld_dict = {'com': 'कॉम', 'in': 'इन', 'org': 'ओआरजी', 'net': 'नेट'}

        self.unit_map = {
            '°C': 'डिग्री सेल्सियस', '°F': 'डिग्री फ़ारेनहाइट',
            'km': 'किलोमीटर', 'cm': 'सेंटीमीटर', 'mm': 'मिलीमीटर',
            'kg': 'किलोग्राम', 'mg': 'मिलीग्राम',
            'GB': 'गीगाबाइट', 'MB': 'मेगाबाइट', 'KB': 'किलोबाइट',
            'Hz': 'हर्ट्ज़', 'MHz': 'मेगाहर्ट्ज़', 'kW': 'किलोवाट', 'kWh': 'किलोवाट घंटा'
        }
        self.ambiguous_units = {'g': 'ग्राम', 'l': 'लीटर', 'L': 'लीटर', 'm': 'मीटर'}

    def _num_to_words(self, num_str: str) -> str:
        try:
            return num2words(int(num_str), lang='hi')
        except Exception:
            return num_str

    def _digit_to_hindi(self, digit: str) -> str:
        d_map = {'0':'शून्य', '1':'एक', '2':'दो', '3':'तीन', '4':'चार', '5':'पाँच',
                 '6':'छह', '7':'सात', '8':'आठ', '9':'नौ'}
        return d_map.get(digit, digit)

    def _latin_to_hindi(self, char: str) -> str:
        char = char.upper()
        c_map = {'A':'ए', 'B':'बी', 'C':'सी', 'D':'डी', 'E':'ई', 'F':'एफ', 'G':'जी',
                 'H':'एच', 'I':'आई', 'J':'जे', 'K':'के', 'L':'एल', 'M':'एम', 'N':'एन',
                 'O':'ओ', 'P':'पी', 'Q':'क्यू', 'R':'आर', 'S':'एस', 'T':'टी', 'U':'यू',
                 'V':'वी', 'W':'डब्लू', 'X':'एक्स', 'Y':'वाई', 'Z':'ज़ेड'}
        return c_map.get(char, char)

    def clean_source_formats(self, text: str) -> str:
        text = html.unescape(text)
        text = text.replace('\ufeff', '').replace('\u200b', '')
        # Line-initial bullets/markers
        text = re.sub(r'(?m)^[\s]*[•◦▪▸→✅❌🔹👉✓✗*>]\s*', '', text)
        return text

    def fix_conjunct_anusvara(self, text: str) -> str:
        def replace_nasal(match):
            nasal = match.group(1)
            halant = match.group(2)
            cons = match.group(3)
            cluster = nasal + halant + cons
            guarded = ['म्ह', 'न्ह', 'न्न', 'न्म', 'न्य', 'ण्ह']
            if cluster in guarded:
                return cluster
            return 'ं' + cons
        return re.sub(r'([ङञणनम])(\u094D)([क-ह])', replace_nasal, text)

    def expand_emails(self, text: str) -> str:
        def email_repl(match):
            user, domain, tld = match.groups()
            user_clean = ""
            for char in user:
                if char == '.': user_clean += " डॉट "
                elif char == '_': user_clean += " अंडरस्कोर "
                elif char.isdigit(): user_clean += f" {self._digit_to_hindi(char)} "
                else: user_clean += char

            d_spoken = self.domain_dict.get(domain.lower(), domain)
            tld_spoken = "डॉट " + self.tld_dict.get(tld.lower(), tld)
            return f" {user_clean.strip()} एट {d_spoken} {tld_spoken} "
        return re.sub(r'([\w.%+\-]+)@([\w.\-]+)\.([a-zA-Z]{2,})', email_repl, text)

    def expand_urls(self, text: str) -> str:
        def url_repl(match):
            url = match.group(1)
            prefix = ""
            if url.startswith('https://'):
                prefix = 'एचटीटीपीएस कोलन स्लैश स्लैश '
                url = url[8:]
            elif url.startswith('http://'):
                prefix = 'एचटीटीपी कोलन स्लैश स्लैश '
                url = url[7:]

            if url.startswith('www.'):
                prefix += 'डब्ल्यूडब्ल्यूडब्ल्यू डॉट '
                url = url[4:]

            parts = re.split(r'([./\-_\?=&])', url)
            spoken = [prefix]

            sym_map = {
                '.': ' डॉट ', '/': ' स्लैश ', '-': ' डैश ',
                '_': ' अंडरस्कोर ', '?': ' क्वेश्चन मार्क ', '=': ' इक्वल्स ', '&': ' एंड '
            }

            for p in parts:
                if not p: continue
                if p in sym_map:
                    spoken.append(sym_map[p])
                elif p.lower() in self.tld_dict:
                    spoken.append(self.tld_dict[p.lower()])
                elif p.lower() in self.domain_dict:
                    spoken.append(self.domain_dict[p.lower()])
                else:
                    spoken.append(p)

            return " " + ''.join(spoken).strip() + " "

        url_regex = r'(https?://[^\s]+)'
        return re.sub(url_regex, url_repl, text)

    def expand_hashtags_mentions(self, text: str) -> str:
        text = re.sub(r'#([\u0900-\u097F]+)', r'\1', text)
        def handle_repl(match):
            handle = match.group(1)
            handle_clean = ''.join([f" {self._digit_to_hindi(c)} " if c.isdigit() else c for c in handle])
            return f" {handle_clean.strip()} "
        text = re.sub(r'@([\w]+)', handle_repl, text)
        text = re.sub(r'#([A-Za-z]+)', r'\1', text)
        return text

    def expand_phones_and_otps(self, text: str) -> str:
        words = text.split()
        for i, word in enumerate(words):
            if re.match(r'^\d{4,8}$', word):
                start = max(0, i - 3)
                context = ' '.join(words[start:i])
                if any(cw in context for cw in self.otp_context_words):
                    words[i] = ' '.join([self._digit_to_hindi(d) for d in word])
            elif re.match(r'^(?:\+91)?\d{10}$', word):
                num_part = word[-10:]
                words[i] = ' '.join([self._digit_to_hindi(d) for d in num_part])
        return ' '.join(words)

    def expand_abbreviations(self, text: str) -> str:
        for abbr, exp in self.hindi_dot_abbrevs.items():
            text = re.sub(r'\b' + abbr + r'\.', exp + ' ', text)

        def latin_abbr(match):
            word = match.group(0)
            if word in self.abbrev_dict:
                return self.abbrev_dict[word]
            return ' '.join([self._latin_to_hindi(c) for c in word])

        text = re.sub(r'\b[A-Z]{2,}\b', latin_abbr, text)
        return text

    def expand_units(self, text: str) -> str:
        sorted_units = sorted(self.unit_map.keys(), key=len, reverse=True)
        for unit in sorted_units:
            escaped_unit = re.escape(unit)
            if re.search(r'\w$', unit):
                pattern = r'(?<=\d)\s*' + escaped_unit + r'\b'
            else:
                pattern = r'(?<=\d)\s*' + escaped_unit
            text = re.sub(pattern, ' ' + self.unit_map[unit] + ' ', text)

        for unit, exp in self.ambiguous_units.items():
            pattern = r'(?<=\d)\s*' + re.escape(unit) + r'\b'
            text = re.sub(pattern, ' ' + exp + ' ', text)

        return text

    def expand_currency(self, text: str) -> str:
        def remove_commas(match):
            return match.group(0).replace(',', '')
        text = re.sub(r'\d{1,3}(?:,\d{2,3})+', remove_commas, text)

        def currency_repl(match):
            sym = match.group(1)
            num = match.group(2)
            c_map = {'₹': 'रुपये', '$': 'डॉलर', '€': 'यूरो', '£': 'पाउंड'}
            return f" {num} {c_map[sym]} "
        text = re.sub(r'([₹$€£])\s*(\d+(?:\.\d+)?)', currency_repl, text)
        return text

    def expand_dates_and_times(self, text: str) -> str:
        def date_repl(match):
            d, m, y = match.groups()
            months = ['', 'जनवरी', 'फरवरी', 'मार्च', 'अप्रैल', 'मई', 'जून',
                      'जुलाई', 'अगस्त', 'सितंबर', 'अक्टूबर', 'नवंबर', 'दिसंबर']
            m_idx = int(m)
            if 1 <= m_idx <= 12:
                return f" {int(d)} {months[m_idx]} {y} "
            return match.group(0)
        text = re.sub(r'\b(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})\b', date_repl, text)

        def time_repl(match):
            h, m = match.groups()
            h_int, m_int = int(h), int(m)
            if m_int == 0:
                return f" {h_int} बजे "
            return f" {h_int} बजकर {m_int} मिनट "
        text = re.sub(r'\b(\d{1,2}):(\d{2})\b', time_repl, text)
        return text

    def expand_percentages(self, text: str) -> str:
        text = re.sub(r'(\d+(?:\.\d+)?)\s*%', r' \1 प्रतिशत ', text)
        return text

    def expand_math(self, text: str) -> str:
        text = text.replace('²', ' वर्ग ').replace('³', ' घन ')
        text = re.sub(r'(?<=\w)\s*\+\s*(?=\w)', ' जमा ', text)
        text = re.sub(r'(?<=\w)\s*-\s*(?=\w)', ' घटा ', text)
        text = re.sub(r'(?<=\w)\s*=\s*(?=\w)', ' बराबर ', text)
        text = re.sub(r'(?<=\w)\s*×\s*(?=\w)', ' गुणा ', text)
        text = re.sub(r'(?<=\w)\s*÷\s*(?=\w)', ' भाग ', text)
        return text

    def expand_general_numbers(self, text: str) -> str:
        def decimal_repl(match):
            int_part = match.group(1)
            frac_part = match.group(2)
            frac_spoken = ' '.join([self._digit_to_hindi(d) for d in frac_part])
            int_spoken = self._num_to_words(int_part)
            return f" {int_spoken} दशमलव {frac_spoken} "
        text = re.sub(r'\b(\d+)\.(\d+)\b', decimal_repl, text)

        def int_repl(match):
            return " " + self._num_to_words(match.group(0)) + " "
        text = re.sub(r'\b\d+\b', int_repl, text)
        return text

    def handle_nukta(self, text: str) -> str:
        # Explicit composition of base + nukta using strict hex codes
        # 0921(ड) + 093c(़) -> 095c(ड़), 0922(ढ) + 093c(़) -> 095d(ढ़)
        comp_map = {
            '\u0915\u093C': '\u0958', # क़
            '\u0916\u093C': '\u0959', # ख़
            '\u0917\u093C': '\u095a', # ग़
            '\u091c\u093C': '\u095b', # ज़
            '\u0921\u093C': '\u095c', # ड़
            '\u0922\u093C': '\u095d', # ढ़
            '\u092b\u093C': '\u095e', # फ़
            '\u092f\u093C': '\u095f', # य़
            # lowercase hex variant
            '\u0915\u093c': '\u0958',
            '\u0916\u093c': '\u0959',
            '\u0917\u093c': '\u095a',
            '\u091c\u093c': '\u095b',
            '\u0921\u093c': '\u095c',
            '\u0922\u093c': '\u095d',
            '\u092b\u093c': '\u095e',
            '\u092f\u093c': '\u095f'
        }
        for decomp, comp in comp_map.items():
            text = text.replace(decomp, comp)

        urdu_nukta_map = {
            '\u0958': '\u0915', # क़ -> क
            '\u0959': '\u0916', # ख़ -> ख
            '\u095a': '\u0917', # ग़ -> ग
            '\u095b': '\u091c', # ज़ -> ज
            '\u095e': '\u092b', # फ़ -> फ
            '\u095f': '\u092f'  # य़ -> य
        }
        for n_char, b_char in urdu_nukta_map.items():
            text = text.replace(n_char, b_char)
        # Stray Nukta removal
        text = text.replace('\u093C', '').replace('\u093c', '')
        return text

    def strip_punctuation(self, text: str) -> str:
        text = re.sub(r'[—–]', ' ', text)
        text = re.sub(r'[।॥\.,\?!\-\'\"\(\)\[\]\{\};:<>_/|\\@#%&*~`^]', ' ', text)
        return text

    def whitespace_clean(self, text: str) -> str:
        return re.sub(r'\s+', ' ', text).strip()

    def validate_normalization(self, text: str):
        words = text.split()
        for word in words:
            if word.startswith('\u0902'):
                logger.warning(f"Word-initial anusvara detected in '{word}' - potential conjunct bug!")

    def normalize(self, text: str) -> str:
        text = self.clean_source_formats(text)
        text = text.translate(self.deva_digit_map)
        text = re.sub(r'<[^>]+>', '', text)
        text = unicodedata.normalize('NFC', text)
        text = text.replace('ॐ', 'ओम').replace('॰', '.')
        text = self.fix_conjunct_anusvara(text)

        text = self.expand_emails(text)
        text = self.expand_urls(text)
        text = self.expand_hashtags_mentions(text)
        text = self.expand_phones_and_otps(text)
        text = self.expand_abbreviations(text)
        text = self.expand_units(text)

        text = self.expand_currency(text)
        text = self.expand_dates_and_times(text)
        text = self.expand_percentages(text)
        text = self.expand_math(text)
        text = self.expand_general_numbers(text)

        text = text.replace('शूंय', 'शून्य')

        text = self.handle_nukta(text)
        text = self.strip_punctuation(text)
        text = self.whitespace_clean(text)

        self.validate_normalization(text)
        return text


# 4. Audio Processing Functions
These functions handle VAD, Resampling, Clipping detection, SNR, and Loudness Normalization.


In [ ]:
import os
import json
import torch
import torchaudio
import librosa
import pyloudnorm as pyln
import soundfile as sf
import numpy as np
from silero_vad import get_speech_timestamps, load_silero_vad
from tqdm import tqdm

# Initialize VAD model on GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
vad_model = load_silero_vad().to(device)

def resample_audio(waveform, orig_sr, target_sr=16000):
    if orig_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_sr, target_sr)
        waveform = resampler(waveform)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    return waveform

def apply_vad(audio_np, target_sr=16000):
    audio_tensor = torch.tensor(audio_np, dtype=torch.float32).to(device)
    speech_timestamps = get_speech_timestamps(
        audio_tensor,
        vad_model,
        threshold=0.01,
        min_speech_duration_ms=300,
        min_silence_duration_ms=500,
        sampling_rate=target_sr
    )
    if not speech_timestamps:
        return None
    speech_segments = [audio_np[seg['start']:seg['end']] for seg in speech_timestamps]
    return np.concatenate(speech_segments)

def detect_clipping(audio_np, threshold=0.95):
    clipped_samples = np.sum(np.abs(audio_np) >= threshold)
    clip_fraction = clipped_samples / len(audio_np) if len(audio_np) > 0 else 0
    return float(clip_fraction)

def calculate_snr(audio_np, n_fft=2048, hop_length=512):
    if len(audio_np) == 0: return None
    D = librosa.stft(audio_np, n_fft=n_fft, hop_length=hop_length)
    power = np.abs(D)**2
    frame_energy = np.sum(power, axis=0)
    if len(frame_energy) == 0: return None
    median_energy = np.median(frame_energy)
    speech = power[:, frame_energy > median_energy]
    noise = power[:, frame_energy <= median_energy]
    if speech.size == 0 or noise.size == 0: return None
    signal_power = np.mean(speech)
    noise_power = np.mean(noise)
    if noise_power == 0: return None
    snr = 10 * np.log10(signal_power / noise_power)
    return float(np.clip(snr, -10, 50))

def normalize_loudness(audio_np, target_sr=16000, target_lufs=-23.0):
    meter = pyln.Meter(target_sr)
    try:
        loudness = meter.integrated_loudness(audio_np)
        norm_audio = pyln.normalize.loudness(audio_np, loudness, target_lufs)
        return norm_audio, float(loudness)
    except:
        return audio_np, None



Using device: cuda


In [ ]:
from huggingface_hub import login
# Generate a Read token in your Hugging Face settings -> Access Tokens
login(token="hf_YOUR_TOKEN_HERE")


# 5. Configuration — Input & Output Paths

**Input**: Rasa HuggingFace dataset loaded directly from Google Drive using `load_from_disk()`  
**Output**: All processed audio and manifests saved under `preprocessed_data_Rasa/` on Drive

> ℹ️ The Rasa folder must be a valid HuggingFace `DatasetDict` saved with `dataset.save_to_disk()`  
> It should contain `dataset_dict.json`, and subdirectories `train/` and `test/` with Arrow shards.


In [ ]:
import os
import shutil
import numpy as np
from datasets import load_dataset

# Set paths
FINAL_DRIVE_DIR = "/content/drive/MyDrive/preprocessed_data_Kathbath/train"

# Duration constraints set to 2.0s - 14.0s
MIN_DURATION     = 2.0
MAX_DURATION     = 14.0
MIN_SNR          = 8.0
MAX_CLIP_FRACTION = 0.01

# --- RUN LIMIT ---
# Set to 1000 for your initial test run.
# Set to None to process the entire dataset.
LIMIT = None

# Custom mappings
gender_map = {
    "f": "Female", "m": "Male", "o": "Other",
    "female": "Female", "male": "Male", "other": "Other"
}
age_group_map = {0: "18-30", 1: "30-45", 2: "45-60", 3: "60+"}
state_map     = {0: "Bihar", 1: "Madhya Pradesh", 2: "Rajasthan", 3: "Uttar Pradesh"}
scenario_map  = {0: "Extempore", 1: "Read"}

print("Loading Kathbath Hindi dataset from Hugging Face in STREAMING mode...")
# streaming=True streams files one by one on-the-fly to prevent Colab disk-full crashes
train_dataset = load_dataset("ai4bharat/Kathbath", "hindi", split="train", streaming=True)
print("✅ Streaming ready!")

# The total split size of Kathbath Hindi train is 91752, but we will stop at 30k target
TOTAL_ROWS = 91752

normalizer = HindiNormalizer()
print("✅ Setup complete. Ready to process train split.")


Loading Kathbath Hindi dataset from Hugging Face in STREAMING mode...


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

✅ Streaming ready!
✅ Setup complete. Ready to process train split.


In [ ]:
print(next(iter(train_dataset)))


{'fname': '844424930703439-329-f.m4a', 'text': 'हालांकि चिकित्सकों ने जेसीओ की मौत हार्ट अटैक से हुई बताया है', 'audio_filepath': <datasets.features._torchcodec.AudioDecoder object at 0x7dad46a040b0>, 'lang': 'hi', 'duration': 4.9923125, 'gender': 'Female', 'speaker_id': 329}


# 6. Pipeline Orchestrator — Train Split (Chunked Output)

Processes only the **`train` split** (~25,713 examples) of the Rasa dataset.  
To avoid freezing Google Drive with 25k tiny file writes, this script writes the `.wav` files to Colab's fast local disk.
Every **5,000 files**, it automatically zips the folder, copies the `.zip` to your Drive (`preprocessed_data_Rasa/train/`), and clears the local space.


In [ ]:
import subprocess
import json
import os
import pandas as pd
import torch
import soundfile as sf
import traceback
from tqdm import tqdm

def text_to_phonemes(text: str, lang: str) -> str:
    if not text: return ''
    try:
        p = subprocess.run(['espeak-ng','-q','-x','-v', lang, text], capture_output=True, text=True, timeout=5)
        return p.stdout.strip()
    except Exception:
        return ''

split_name = "train"
audio_out_dir = os.path.join(FINAL_DRIVE_DIR, "audio", "normalized")
manifest_path = os.path.join(FINAL_DRIVE_DIR, "final_manifest.jsonl")
parquet_path  = os.path.join(FINAL_DRIVE_DIR, "final_manifest.parquet")
report_path   = os.path.join(FINAL_DRIVE_DIR, "quality_report.txt")

os.makedirs(audio_out_dir, exist_ok=True)

processed_records = []
processed_ids = set()

# Safe resume log
reject_reasons = {
    "vad_failure": 0,
    "duration_out_of_bounds": 0,
    "excessive_clipping": 0,
    "low_snr": 0,
    "processing_error": 0
}

# SAFE RESUME LOGIC: Load already processed records from manifest
if os.path.exists(manifest_path):
    print(f"Found existing manifest. Loading processed IDs for safe resume...")
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    record = json.loads(line)
                    processed_ids.add(record["id"])
                    processed_records.append(record)
                except Exception:
                    pass
    print(f"Resuming safely. Skipping {len(processed_ids)} already processed records.")

# If we already have 30,000 processed records, we can finish immediately
if len(processed_records) >= 30000:
    print(f"🎉 Target reached! You already have {len(processed_records)} processed records in your manifest.")

# Set progress bar target
tqdm_total = 30000

# Open in APPEND mode
with open(manifest_path, "a", encoding="utf-8") as manifest_f:
    for idx, example in enumerate(tqdm(train_dataset, desc=split_name, total=tqdm_total)):
        # Stop once we have reached 30,000 successfully processed (passed) samples
        if len(processed_records) >= 30000:
            print(f"\n🎉 Target of 30,000 processed samples successfully reached!")
            break

        # Stop if we hit the inspection limit (if set)
        if LIMIT is not None and idx >= LIMIT:
            print(f"\nReached {LIMIT} sample limit.")
            break

        # Kathbath naming prefix
        file_id = f"kathbath_{split_name}_{idx:06d}"

        # Check safe resume
        if file_id in processed_ids:
            continue

        try:
            # Access the audio decoder column
            audio_dict  = example["audio_filepath"]
            audio_array = audio_dict["array"]
            orig_sr     = audio_dict["sampling_rate"]

            waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)

            target_sr    = 16000
            waveform_16k = resample_audio(waveform, orig_sr, target_sr)
            audio_np     = waveform_16k.squeeze().numpy()
            orig_duration = len(audio_np) / target_sr

            vad_audio = apply_vad(audio_np, target_sr)
            if vad_audio is None:
                reject_reasons["vad_failure"] += 1
                continue

            new_duration = len(vad_audio) / target_sr
            if new_duration < MIN_DURATION or new_duration > MAX_DURATION:
                reject_reasons["duration_out_of_bounds"] += 1
                continue

            clip_frac = detect_clipping(vad_audio)
            if clip_frac > MAX_CLIP_FRACTION:
                reject_reasons["excessive_clipping"] += 1
                continue

            snr_val = calculate_snr(vad_audio)
            if snr_val is None or snr_val < MIN_SNR:
                reject_reasons["low_snr"] += 1
                continue

            norm_audio, lufs = normalize_loudness(vad_audio, target_sr)
            if lufs is None:
                norm_audio = vad_audio

            # Fetch transcription (using 'text' key)
            raw_text  = example.get("text", "")
            norm_text = normalizer.normalize(raw_text)

            lang_code = "hi"
            phonemes_out = text_to_phonemes(norm_text, lang_code)

            out_filename = f"{file_id}.wav"
            out_filepath = os.path.join(audio_out_dir, out_filename)

            sf.write(out_filepath, norm_audio, target_sr)

            # Retrieve original path or filename safely without calling .get() on AudioDecoder
            try:
                orig_path = audio_dict["path"]
            except Exception:
                orig_path = example.get("fname", "")

            # Extract metadata values directly
            speaker_id_val = example.get("speaker_id", "")
            gender_val = example.get("gender", "Other")

            record = {
                "id":                    file_id,
                "split":                 split_name,
                "category":              "Read",
                "text":                  raw_text,
                "normalized_text":       norm_text,
                "phonemes":              phonemes_out,
                "source":                "Kathbath",
                "original_audio_filepath": orig_path,
                "audio_filepath":         out_filepath,
                "duration":              round(new_duration, 3),
                "original_duration":     round(orig_duration, 3),
                "language":              example.get("lang", "hi"),
                "script":                "Devanagari",
                "speaker_id":            str(speaker_id_val),
                "gender":                gender_val,
                "age_group":             "Unknown",
                "district":              "",
                "state":                 "Unknown",
                "snr":                   round(snr_val, 2),
                "clip_fraction":         round(clip_frac, 4),
                "loudness_lufs":         round(lufs, 2) if lufs is not None else None
            }

            manifest_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            processed_records.append(record)
            processed_ids.add(file_id)

        except Exception as e:
            if reject_reasons["processing_error"] < 5:
                print(f"\nError at index {idx}: {e}")
                traceback.print_exc()
            elif reject_reasons["processing_error"] == 5:
                print("... suppressing further processing error logs ...")
            reject_reasons["processing_error"] += 1

print(f"\nGenerating Parquet Database...")
if processed_records:
    df = pd.DataFrame(processed_records)
    df.to_parquet(parquet_path, engine='pyarrow', index=False)

print(f"Generating Quality Report...")
total_processed = len(processed_records)
session_rejected = sum(reject_reasons.values())

# Compute distributions
dist_keys = ['gender', 'age_group', 'state', 'category']
dist_stats = []
for key in dist_keys:
    counts = {}
    for r in processed_records:
        val = r.get(key, "Unknown")
        if not val: val = "Unknown"
        counts[val] = counts.get(val, 0) + 1
    dist_stats.append(f"{key.capitalize()} distribution:")
    for val, count in sorted(counts.items(), key=lambda x: -x[1]):
        pct = (count / total_processed * 100) if total_processed > 0 else 0
        dist_stats.append(f" - {val}: {count} ({pct:.2f}%)")
    dist_stats.append("")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("=========================================\n")
    f.write("Quality Filtering Summary Report\n")
    f.write("=========================================\n")
    f.write(f"Total Passed Items (All Sessions): {total_processed}\n")
    f.write(f"Items Rejected (Current Session):  {session_rejected}\n\n")

    f.write("Rejection Breakdown (Current Session):\n")
    f.write(f" - VAD Failed (No Speech): {reject_reasons['vad_failure']}\n")
    f.write(f" - Duration Out of Bounds: {reject_reasons['duration_out_of_bounds']}\n")
    f.write(f" - Excessive Clipping:     {reject_reasons['excessive_clipping']}\n")
    f.write(f" - Low SNR (Too Noisy):    {reject_reasons['low_snr']}\n")
    f.write(f" - Processing Errors:      {reject_reasons['processing_error']}\n\n")

    if total_processed > 0:
        avg_snr = sum([r['snr'] for r in processed_records]) / total_processed
        f.write(f"Average SNR (Passed):      {avg_snr:.2f} dB\n")
    f.write("=========================================\n\n")
    f.write("Metadata Distributions (Passed Items)\n")
    f.write("=========================================\n")
    f.write("\n".join(dist_stats))

print(f"\n✅ [{split_name}] Done!")


Found existing manifest. Loading processed IDs for safe resume...
Resuming safely. Skipping 0 already processed records.


train:   0%|          | 54/30000 [00:11<1:28:45,  5.62it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
train:   0%|          | 62/30000 [00:12<1:17:09,  6.47it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
train:   2%|▏         | 523/30000 [01:22<1:46:44,  4.60it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
train:   3%|▎         | 784/30000 [02:02<1:34:00,  5.18it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
train:   3%|▎         | 850/30000 [02:12<1:10:11,  6.92it/s]/usr/local/lib


🎉 Target of 30,000 processed samples successfully reached!

Generating Parquet Database...
Generating Quality Report...

✅ [train] Done!
